# 🛡️ ZoneShield AI — YOLOv8 Conveyor Belt Training
**Profile:** Competition-grade (300 epochs, medium model)

**Dataset:** Roboflow conveyor-hhrzw v3 (CC BY 4.0)

**Classes:** cardboard_box, conveyor, kartonbox

⚡ **Runtime → Change runtime type → T4 GPU**

In [ ]:
# 1. Install dependencies
!pip install ultralytics roboflow -q

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


In [ ]:
# 2. Download dataset from Roboflow
from roboflow import Roboflow
rf = Roboflow(api_key='YOUR_ROBOFLOW_API_KEY')  # Get from roboflow.com/settings
project = rf.workspace('conveyor-550m0').project('conveyor-hhrzw')
version = project.version(3)
dataset = version.download('yolov8')
print(f'Dataset path: {dataset.location}')


In [ ]:
# 2b. ALTERNATIVE: Upload your local dataset zip
# Upload conveyor.v3i.yolov8.zip via Colab sidebar, then:
# !unzip -q conveyor.v3i.yolov8.zip -d /content/conveyor_dataset
# dataset_location = '/content/conveyor_dataset'


In [ ]:
# 3. Train YOLOv8 Segmentation Model
from ultralytics import YOLO

model = YOLO('yolov8m-seg.pt')

results = model.train(
    data=dataset.location + '/data.yaml',
    epochs=300,
    imgsz=640,
    batch=8,  # Colab GPU can handle larger batch
    patience=50,
    lr0=0.005,
    lrf=0.0005,
    warmup_epochs=10,
    cos_lr=True,
    optimizer='AdamW',
    weight_decay=0.001,
    device=0,
    amp=True,
    seed=42,
    # Heavy augmentation for class imbalance
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=15.0, translate=0.15, scale=0.5,
    shear=5.0, perspective=0.001,
    flipud=0.3, fliplr=0.5,
    mosaic=1.0, mixup=0.15,
    copy_paste=0.2,  # Critical for seg with class imbalance
    erasing=0.1,
    close_mosaic=30,
    plots=True, save=True, val=True,
    name='zoneshield-competition',
)


In [ ]:
# 4. Validate on test set
best = YOLO('runs/segment/zoneshield-competition/weights/best.pt')
metrics = best.val(data=dataset.location + '/data.yaml', split='test')
print(f'mAP50: {metrics.seg.map50:.4f}')
print(f'mAP50-95: {metrics.seg.map:.4f}')


In [ ]:
# 5. Export to ONNX (for OpenCV DNN)
onnx_path = best.export(
    format='onnx',
    imgsz=640,
    opset=12,
    simplify=True,
    dynamic=False,
    half=False,
)
print(f'ONNX exported: {onnx_path}')


In [ ]:
# 6. Download the trained model
from google.colab import files
files.download('runs/segment/zoneshield-competition/weights/best.pt')
files.download(onnx_path)

# Also zip all results
!zip -r zoneshield_results.zip runs/segment/zoneshield-competition/
files.download('zoneshield_results.zip')


In [ ]:
# 7. Quick inference test
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
import glob

test_imgs = glob.glob(dataset.location + '/test/images/*.jpg')[:4]
fig, axes = plt.subplots(1, len(test_imgs), figsize=(20, 5))
for ax, img_path in zip(axes, test_imgs):
    results = best.predict(img_path, conf=0.25, verbose=False)
    annotated = results[0].plot()
    ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    ax.axis('off')
    ax.set_title(Path(img_path).stem[:20])
plt.tight_layout()
plt.savefig('inference_preview.png', dpi=150)
plt.show()
